# M10 — SMPL body through the static map (Parts A / A.5 / B)

Thin GPU runner for `tracking/smpl_person.py`. Logic lives in the fork; this notebook only clones, installs, mounts Drive, sets paths, runs, and displays.

- **Part A** — render BEHAVE **GT** person (from `person/fit02/person.ply` — no SMPL model download) moving through the reconstructed static surfel map.
- **Part A.5** — swap GT for **MAMMA-estimated SMPL-X** (run MAMMA on the 4 views first; §6).
- **Part B** — root-trajectory forecast, **ADE/FDE vs constant-velocity**.

Paths in §4 are pre-filled for **Date03_Sub05_boxtiny**. Run on a GPU runtime (A100), from the **gmail** account whose Drive holds the data. NOTE: boxtiny is in-place manipulation → Part B numbers are a plumbing check; use a walking sequence for the real prediction result.

## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2+3. Mount Drive + clone the fork + build CUDA submodules + deps
Same proven setup as the `behave_deform_run` notebooks, plus `plyfile`/`trimesh` (read `person.ply`) and `smplx` (only if you fall back to posing params). The committed `render()` already accepts the means/rotation overrides `render_composite` uses — no runtime patch needed.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys
os.chdir('/content')
if not os.path.isdir('/content/gaussian-surfel-local-map'):
    !git clone --recursive https://github.com/steptang/gaussian-surfel-local-map.git
sys.path.insert(0, '/content/gaussian-surfel-local-map')
os.chdir('/content/gaussian-surfel-local-map')
!git pull -q   # need the branch with tracking/smpl_person.py
!pip install -q plyfile open3d scikit-learn scipy mediapy imageio imageio-ffmpeg trimesh opencv-python-headless smplx
!pip install -q --no-build-isolation ./submodules/diff-surfel-rasterization
!pip install -q --no-build-isolation ./submodules/simple-knn
print('ready')

## 4. Paths (pre-filled for boxtiny; outputs → Drive)
- `CONV_ROOT` — converted per-timestep scenes (`timestep_*`).
- `SMPL_ROOT` (Part A) — raw BEHAVE sequence (`t*.000/person/fit02/person.ply`); same sequence as `CONV_ROOT`.
- `MAMMA_ROOT` / `MAMMA_MODEL_DIR` (Part A.5) — MAMMA's per-frame exports + the SMPL-X model dir on Drive (from the `run_mamma_behave` notebook). No `/content/mamma` here — this is a separate session.
- Outputs go straight to Drive (`OUT_A`, `OUT_A5`) so a timeout doesn't lose them.

In [ ]:
DRIVE = '/content/drive/MyDrive/Research'
CONV_ROOT      = f'{DRIVE}/2dgs_output/datasets/behave_Date03_Sub05_boxtiny'
SMPL_ROOT      = f'{DRIVE}/datasets/behave/sequences/Date03_Sub05_boxtiny'   # Part A: raw BEHAVE person.ply
N_SELECT       = 24          # converter's N_TIMESTEPS (SEL = sorted(t*.000)[::stride][:N])
# Part A.5 (MAMMA): estimated SMPL-X exports + the SMPL-X model dir (both on Drive, from run_mamma_behave)
MAMMA_ROOT      = f'{DRIVE}/datasets/mamma_exports/Date03_Sub05_boxtiny'
MAMMA_MODEL_DIR = f'{DRIVE}/2dgs_output/mamma_run/body_models/smplx_locked_head'
# outputs -> Drive (durable across timeouts)
OUT_A  = f'{DRIVE}/2dgs_output/results/smpl_person/partA_gt'
OUT_A5 = f'{DRIVE}/2dgs_output/results/smpl_person/partA5_mamma'
import os
for d in (OUT_A, OUT_A5): os.makedirs(d, exist_ok=True)
assert os.path.isdir(CONV_ROOT), CONV_ROOT; assert os.path.isdir(SMPL_ROOT), SMPL_ROOT
print('paths OK')

## 5. Part A — GT person through the static map (+ Part B baseline)
First run: check the printed `timestep → raw frame` mapping, then eyeball `compare.png`. If the body floats / is rotated / wrong scale, add `--align_t x y z` / `--align_s` and re-run.

In [ ]:
!python -m tracking.smpl_person \
  --conv_root "$CONV_ROOT" --smpl_root "$SMPL_ROOT" \
  --smpl_source gt --view 0 --n_select $N_SELECT --out "$OUT_A" \
  --pred_hist 8 --pred_horizon 8

In [ ]:
import json
from IPython.display import Image, Video, display
OUT = OUT_A
print(json.dumps(json.load(open(f'{OUT}/metrics.json')), indent=2))
print(json.dumps(json.load(open(f'{OUT}/prediction.json')), indent=2))
display(Image(f'{OUT}/compare.png'))
display(Video(f'{OUT}/sequence.mp4', embed=True, width=720))

## 6. Part A.5 — MAMMA-estimated SMPL-X through the static map
Needs the MAMMA exports + SMPL-X model dir on Drive (produced by the **`M10 - run_mamma_behave`** notebook §7). Runs the same pipeline with `--smpl_source mamma`. Since MAMMA used your BEHAVE calibration, its body is already in the surfel-map world frame — if it lands on the person, the calibration convention was right; if not, re-run MAMMA §5 with `--extrinsics cam2world` / `--quat_order xyzw`.

In [ ]:
!python -m tracking.smpl_person \
  --conv_root "$CONV_ROOT" --smpl_root "$MAMMA_ROOT" --smpl_model_dir "$MAMMA_MODEL_DIR" \
  --smpl_source mamma --view 0 --n_select $N_SELECT --out "$OUT_A5" \
  --pred_hist 8 --pred_horizon 8

## 7. Compare GT (A) vs MAMMA (A.5)
(Outputs already persist on Drive under `OUT_A` / `OUT_A5`.)

import json
for tag, OUT in [('GT (A)', OUT_A), ('MAMMA (A.5)', OUT_A5)]:
    try:
        m = json.load(open(f'{OUT}/metrics.json')); p = json.load(open(f'{OUT}/prediction.json'))
        print(f"{tag:12s} person_psnr_mean={m['person_psnr_mean']:.2f}  "
              f"const_vel_ADE={p['const_vel_ADE']:.4f}  FDE={p['const_vel_FDE']:.4f}")
    except FileNotFoundError:
        print(f"{tag:12s} not run yet ({OUT})")

In [ ]:
SAVE_TO = f'{DRIVE}/2dgs_output/results/smpl_person'
!mkdir -p "$SAVE_TO" && cp -r "$OUT_A" "$SAVE_TO/" && echo saved to "$SAVE_TO"